In [ ]:
import os
print(os.listdir("/kaggle/input/competitions"))

In [ ]:
import os
import ast
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image

import albumentations as A
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CLASS_NAME = "wheat"
CLASS_ID = 0

# ----- Data paths -----
# Adjust DATA_DIR to wherever the competition data was downloaded/extracted.
# Expected structure:
#   DATA_DIR/train.csv
#   DATA_DIR/train/*.jpg
#   DATA_DIR/test/*.jpg
DATA_DIR = Path("/kaggle/input/competitions/global-wheat-detection")          # e.g. Path("/kaggle/input/global-wheat-detection")
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMG_DIR = DATA_DIR / "train"
TEST_IMG_DIR = DATA_DIR / "test"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
YOLO_DIR = OUTPUT_DIR / "yolo_dataset"

assert TRAIN_CSV.exists(), f"train.csv not found at {TRAIN_CSV.resolve()} - update DATA_DIR"
print("Using data directory:", DATA_DIR.resolve())

In [ ]:
all_train_images = sorted(p.stem for p in TRAIN_IMG_DIR.glob("*.jpg"))

def parse_bbox(bbox_str):
    """Parse the string-encoded bbox '[xmin, ymin, w, h]' into floats."""
    return ast.literal_eval(bbox_str)

df_raw = pd.read_csv(TRAIN_CSV)
df = df_raw.copy()
bbox_arr = np.array(df["bbox"].apply(parse_bbox).tolist())
df["x_min"] = bbox_arr[:, 0]
df["y_min"] = bbox_arr[:, 1]
df["box_width"] = bbox_arr[:, 2]
df["box_height"] = bbox_arr[:, 3]

images_with_boxes = set(df["image_id"].unique())
images_without_boxes = sorted(set(all_train_images) - images_with_boxes)

In [ ]:
SMALL_AREA_RATIO_THRESH = 0.0005   # box covers < 0.05% of its image
LARGE_AREA_RATIO_THRESH = 0.15     # box covers > 15% of its image
df["area"] = df["box_width"] * df["box_height"]
df["image_area"] = df["width"] * df["height"]
df["area_ratio"] = df["area"] / df["image_area"]
NEG_DIM_MASK = (df["box_width"] <= 0) | (df["box_height"] <= 0)
SMALL_MASK = (~NEG_DIM_MASK) & (df["area_ratio"] < SMALL_AREA_RATIO_THRESH)
LARGE_MASK = (~NEG_DIM_MASK) & (df["area_ratio"] > LARGE_AREA_RATIO_THRESH)
NORMAL_MASK = ~(NEG_DIM_MASK | SMALL_MASK | LARGE_MASK)

In [ ]:
clean_df = df.copy()
clean_df["x_max"] = clean_df["x_min"] + clean_df["box_width"]
clean_df["y_max"] = clean_df["y_min"] + clean_df["box_height"]

clean_df["is_negative_dim"] = NEG_DIM_MASK
clean_df["is_small_outlier"] = SMALL_MASK
clean_df["is_large_outlier"] = LARGE_MASK
clean_df["is_outlier"] = NEG_DIM_MASK | SMALL_MASK | LARGE_MASK
clean_df["use_for_training"] = ~(clean_df["is_outlier"])

attribute_cols = [
    "image_id", "width", "height", "source",
    "x_min", "y_min", "box_width", "box_height", "x_max", "y_max", "is_outlier",
    "use_for_training"
]
clean_df = clean_df[attribute_cols]

In [ ]:
from sklearn.model_selection import train_test_split

# --- Build an 80/10/10 split, stratified by source ---
image_source_df = clean_df[["image_id", "source"]].drop_duplicates()

train_ids, temp_ids = train_test_split(
    image_source_df["image_id"], test_size=0.20,
    stratify=image_source_df["source"], random_state=RANDOM_SEED,
)
temp_source = image_source_df.set_index("image_id").loc[temp_ids, "source"]
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, stratify=temp_source, random_state=RANDOM_SEED,
)  # 0.5 of the 20% held out -> 10% val, 10% test

# images with no boxes have no known source - split them the same way, unstratified
no_box_ids = sorted(set(all_train_images) - set(image_source_df["image_id"]))
nb_train, nb_temp = train_test_split(no_box_ids, test_size=0.20, random_state=RANDOM_SEED)
nb_val, nb_test = train_test_split(nb_temp, test_size=0.50, random_state=RANDOM_SEED)

split_lookup = {}
for img_id in list(train_ids) + nb_train:
    split_lookup[img_id] = "train"
for img_id in list(val_ids) + nb_val:
    split_lookup[img_id] = "val"
for img_id in list(test_ids) + nb_test:
    split_lookup[img_id] = "test"

print(pd.Series(split_lookup).value_counts())

In [ ]:
YOLO_IMG_DIR = {s: YOLO_DIR / "images" / s for s in ["train", "val", "test"]}
YOLO_LBL_DIR = {s: YOLO_DIR / "labels" / s for s in ["train", "val", "test"]}
for d in list(YOLO_IMG_DIR.values()) + list(YOLO_LBL_DIR.values()):
    d.mkdir(parents=True, exist_ok=True)

def to_yolo_line(row, img_w, img_h):
    x_center = (row.x_min + row.box_width / 2) / img_w
    y_center = (row.y_min + row.box_height / 2) / img_h
    w = row.box_width / img_w
    h = row.box_height / img_h
    x_center, y_center, w, h = (float(np.clip(v, 0, 1)) for v in (x_center, y_center, w, h))
    return f"{CLASS_ID} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"

boxes_for_yolo = clean_df[clean_df["use_for_training"]]

for img_id in all_train_images:
    split = split_lookup.get(img_id, "train")  # safe now: "train"/"val"/"test" all exist as keys
    rows = boxes_for_yolo[boxes_for_yolo["image_id"] == img_id]

    img_w, img_h = 1024, 1024
    if len(rows) > 0:
        img_w = int(rows.iloc[0]["width"])
        img_h = int(rows.iloc[0]["height"])

    lines = [to_yolo_line(r, img_w, img_h) for r in rows.itertuples()]
    (YOLO_LBL_DIR[split] / f"{img_id}.txt").write_text("\n".join(lines))

    src_img = TRAIN_IMG_DIR / f"{img_id}.jpg"
    dst_img = YOLO_IMG_DIR[split] / f"{img_id}.jpg"
    if not dst_img.exists():
        try:
            os.link(src_img, dst_img)
        except OSError:
            shutil.copy(src_img, dst_img)

data_yaml = f"""path: {YOLO_DIR.resolve()}
train: images/train
val: images/val
test: images/test
names:
  0: {CLASS_NAME}
"""
(YOLO_DIR / "data.yaml").write_text(data_yaml)

for split in ["train", "val", "test"]:
    n_img = len(list(YOLO_IMG_DIR[split].glob("*.jpg")))
    n_lbl = len(list(YOLO_LBL_DIR[split].glob("*.txt")))
    print(f"{split:5s}: {n_img} images, {n_lbl} labels")

In [ ]:
!pip install ultralytics
from ultralytics import YOLO

# Load the model
model = YOLO("yolo26s.pt")

# Run the automated hyperparameter search
model.tune(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=10,         # Keep low (10) just to find the best settings quickly
    iterations=10,     # Tests 10 different combinations
    imgsz=480,         # Keep low (480) to make the search run much faster
    batch=32,          # High batch for speed
    optimizer="AdamW", # Good optimizer for tuning
    plots=False,       # Save time by not generating plots during search
    save=False,        # Don't save the 10 temporary models, just the results
    project="global_wheat",
    name="hyperparameter_search",
    exist_ok=True,
)